In [3]:
import numpy as np
import pandas as pd
import io 
import os

import ftplib
from ftplib import FTP

## Conectando ao Servidor FTP

In [4]:
ftp_server = "ftp.dadosabertos.ans.gov.br"

In [5]:
class SmartFTP(FTP):
    def makepasv(self):
        invalidhost, port = super(SmartFTP, self).makepasv()
        return self.host, port

        


In [6]:
ftp = SmartFTP(ftp_server)

### Login

In [7]:
print(ftp.login())

ftp.encoding = 'Utf-8'

230 Login successful.


In [8]:
print(ftp.getwelcome())

220 'Arquivos de Dados Publicos ANS'


### Verificando os diretórios existentes

In [9]:
print(ftp.dir())

drwxrwxr-x    4 0        0              38 Aug 25  2023 FTP
drwxr-sr-x    2 0        0              17 Jan 25 12:45 dev
None


### Navegação dentro do servidor FTP

In [10]:
# Inserir o nome do diretório a ser acessado
ftp.cwd('FTP')
ftp.dir()

drwxrwxr-x    3 1001     1001           24 Aug 21  2023 Base_de_dados
drwxrwxr-x   53 1001     1001         4096 Jan 17 14:00 PDA


In [11]:
#Nivel 3 da arvore do FTP
ftp.cwd('PDA')

print("Arquivos disponiveis")
print("***********************")
print()
print('\n'.join(ftp.nlst()))
print()
print("***********************")

Arquivos disponiveis
***********************

Caderno_SS
IAP
IGR
PFA
RPC
SIP
TISS
agenda_de_autoridades
area_comercializacao_planos_ntrp
beneficiarios_identificados_sus_abi
beneficiarios_vinculos_tipo_contratacao_vda
caderno_de_informacao
caracteristicas_produtos_saude_suplementar-008
dados_consolidados_da_saude_suplementar
dados_de_beneficiarios_por_operadora
dados_de_beneficiarios_por_regiao_geografica
dados_ressarcimento_SUS_operadora_planos_saude
dataset_teste
demandas_dos_consumidores_nip
demandas_dos_consumidores_reclamacao_beneficiarios
demonstracoes_contabeis
faixa_de_preco
glossario_saude_suplementar
hc_ressarcimento_sus
historico_idss-020
historico_planos_saude
informacoes_consolidadas_de_beneficiarios-024
monitoramento_garantia_atendimento
nota_tecnica_ntrp_vcm_faixa_etaria
operadoras_acreditadas
operadoras_de_plano_de_saude_ativas
operadoras_de_plano_de_saude_canceladas
operadoras_e_prestadores_nao_hospitalares
painel_de_precificacao
penalidades_aplicadas_a_operadoras
peona

## Arquivo: Operadoras ativas

In [12]:
ftp.cwd('operadoras_de_plano_de_saude_ativas')
ftp.nlst()

['Relatorio_cadop.csv', 'dicionario_de_dados_das_operadoras_ativas.ods']

In [13]:
file_csv = 'Relatorio_cadop.csv'
file_dicio = 'dicionario_de_dados_das_operadoras_ativas.ods'

In [14]:
def lerfile_ftp(file):
    download_file = io.BytesIO()

    ftp.retrbinary('RETR ' + str(file), download_file.write)

    download_file.seek(0)
    return download_file

### Lendo arquivo para dataframe

In [15]:

ope_ativas = pd.read_fwf(lerfile_ftp(file_csv), encoding = 'utf-8', sep = ';')

linhas = [v for v in [ope_ativas.values]][0]
oper_ativas_list = [l.split(';') for l in linhas[:, 0] ]
colunas = [c.split(';') for c in ope_ativas.columns.tolist()]

oper_ativas = pd.DataFrame(data = oper_ativas_list, columns = colunas[0])

for c in oper_ativas.columns:
    
    oper_ativas[c] = oper_ativas[c].str.replace('"', "").replace("  ", " ")
    oper_ativas[c] = oper_ativas[c].squeeze()
    

In [16]:
#download do dicionario de dados operaa ativas

dicio = pd.read_excel(lerfile_ftp(file_dicio), engine='odf')

colunas = dicio.dropna().reset_index(drop = True).values.tolist()[0]

dicio.columns = colunas
dicio.dropna(inplace = True)
dicio.reset_index(drop = True, inplace = True)
dicio = dicio.loc[1:, :]

In [17]:
dicio

,Nome do Campo,Tipo,Tamanho,Descrição
1,REGISTRO_OPERADORA,Texto,6,Registro de operadora de plano privado de assi...
2,CNPJ,Texto,14,CNPJ da Operadora
3,Razao_Social,Texto,140,Razão Social da Operadora
4,Nome_Fantasia,Texto,140,Nome Fantasia da Operadora
5,Modalidade,Texto,2,Classificação das operadoras deplanos privados...
6,Logradouro,Texto,40,Endereço da Sede da Operadora
7,Número,Texto,20,Número do Endereço da Sede da Operadora
8,Complemento,Texto,40,Complemento do Endereço da Sede da Operadora
9,Bairro,Texto,30,Bairro do Endereço da Sede da Operadora
10,Cidade,Texto,30,Cidade do Endereço da Sede da Operadora


#### Data Discovery

In [18]:
oper_ativas.head()

,Registro_ANS,CNPJ,Razao_Social,Nome_Fantasia,Modalidade,Logradouro,Numero,Complemento,Bairro,Cidade,UF,CEP,DDD,Telefone,Fax,Endereco_eletronico,Representante,Cargo_Representante,Regiao_de_Comercializacao,Data_Registro_ANS
0,419761,19541931000125,18 DE JULHO ADMINISTRADORA DE BENEFÍCIOS LTDA,,Administradora de Benefícios,RUA CAPITÃO MEDEIROS DE REZENDE,274,,PRAÇA DA BANDEIRA,Além Paraíba,MG,36660000,32,34624649,,contabilidade@cbnassessoria.com.br,LUIZ HENRIQUE MARENDINO GONÇALVES,SÓCIO ADMINISTRADOR,6,2015-05-19
1,421545,22869997000153,2B ODONTOLOGIA OPERADORA DE PLANOS ODONTOLÓGIC...,,Odontologia de Grupo,RUA CATÃO,128,SALA 126,VILA ROMANA,São Paulo,SP,05049000,11,34415852,,labmarisol@gmail.com,MARISOL BECHELLI,SÓCIO ADMINISTRADORA,4,2019-06-13
2,421421,27452545000195,2CARE OPERADORA DE SAÚDE LTDA.,,Medicina de Grupo,RUA: BERNARDINO DE CAMPOS,230,1º ANDAR,CENTRO,Campinas,SP,13010151,19,37901224,,ans.plano@hospitalcare.com.br,RODRIGO PINHO RIBEIRO,REPRESENTANTE,5,2018-10-09
3,422908,41788751000100,3S ADMINISTRADORA DE BENEFICIOS LTDA,3S ADMINISTRADORA DE BENEFICIOS,Administradora de Benefícios,RUA ITÁLIA,33,SALA 6,JARDIM BONFIGLIOLI,Jundiaí,SP,13207280,11,42263247,,pietrorocchi37@gmail.com,ANAMELIA MONTEIRO GUERRA ROCCHI,SÓCIO ADMINISTRADOR,6,2021-07-12
4,418030,13138885000131,A.P.S. ADMINISTRADORA DE BENEFÍCOS LTDA.,A.P.S. SAÚDE.,Administradora de Benefícios,RUA VOLUNTÁRIOS DA PÁTRIA,2525,CONJUNTO 143 - SALA 01,SANTANA,São Paulo,SP,02401000,11,45223468,,diretoria@apssaude.com.br,PERCÍVEL GAETA,SóCIO-ADMINISTRADOR E REPRESEN,4,2011-05-05


In [19]:
oper_ativas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1110 entries, 0 to 1109
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Registro_ANS               1110 non-null   object
 1   CNPJ                       1110 non-null   object
 2   Razao_Social               1110 non-null   object
 3   Nome_Fantasia              1110 non-null   object
 4   Modalidade                 1110 non-null   object
 5   Logradouro                 1110 non-null   object
 6   Numero                     1110 non-null   object
 7   Complemento                1110 non-null   object
 8   Bairro                     1110 non-null   object
 9   Cidade                     1110 non-null   object
 10  UF                         1110 non-null   object
 11  CEP                        1110 non-null   object
 12  DDD                        1110 non-null   object
 13  Telefone                   1110 non-null   object
 14  Fax     

In [20]:
oper_ativas['Registro_ANS'].value_counts().head(10)

#Não há operadoras duplicadas

Registro_ANS
419761    1
343889    1
343765    1
344141    1
350346    1
323357    1
334561    1
303976    1
304468    1
345598    1
Name: count, dtype: int64

In [21]:
oper_ativas['Data_Registro_ANS'].value_counts()

Data_Registro_ANS
1998-12-17    100
1998-12-18     75
1998-12-21     70
1998-12-16     68
1998-12-22     63
             ... 
2009-03-02      1
2021-10-04      1
2001-05-25      1
1999-01-20      1
1999-08-23      1
Name: count, Length: 423, dtype: int64

In [22]:
oper_ativas.isna().sum()

Registro_ANS                  0
CNPJ                          0
Razao_Social                  0
Nome_Fantasia                 0
Modalidade                    0
Logradouro                    0
Numero                        0
Complemento                   0
Bairro                        0
Cidade                        0
UF                            0
CEP                           0
DDD                           0
Telefone                      0
Fax                           0
Endereco_eletronico           0
Representante                 0
Cargo_Representante           0
Regiao_de_Comercializacao     9
Data_Registro_ANS            11
dtype: int64

In [23]:
oper_ativas[oper_ativas['Data_Registro_ANS'].isna()]

,Registro_ANS,CNPJ,Razao_Social,Nome_Fantasia,Modalidade,Logradouro,Numero,Complemento,Bairro,Cidade,UF,CEP,DDD,Telefone,Fax,Endereco_eletronico,Representante,Cargo_Representante,Regiao_de_Comercializacao,Data_Registro_ANS
105,408263,71753297000104,ASSOCIAÇÃO POLICIAL DE ASSISTENCIA À SAUDE DE ...,APAS - ASSOCIAÇÃO POLICIAL DE ASSISTÊCIA À SAÚ...,Autogestão,AV. PRESIDENTE JOÃO BELCHIOR MARQUES GOU,401,,PARQUE NAÇÕES,São João da Boa Vista,SP,13870579,19,36331494,36338415,apas.sjoao@uol.com.br,CELSO AUGUSTO LUCIO,DIRETOR PRESIDENTE,5,None
200,417661,11996146000155,CAIXA DE ASSISTÊNCIA À SAÚDE DO SINDICATO DOS ...,FISCO SAÚDE-PE,Autogestão,RUA DA AURORA,1443,SALA 1,SANTO AMARO,Recife,PE,50040090,81,31267702,31267711,fiscosaude@fiscosaudepe.com.br,MARIA FERNANDA TORRES,Gerente de Operaç,None,None
492,422851,36765005000152,ODONTO MAIS BRASIL OPERADORA DE PLANOS ODONTOL...,ODONTO MAIS BRASIL OPERADORA DE PLANOS ODONTOL...,Odontologia de Grupo,AV RIO JUTAI,26,QD 36 - COND. RESID. ISAIAS VIEIRA,NOSSA SENHORA DAS GRACAS,Manaus,AM,69053020,92,93791095,,administrativo@maisodontobrasil.com.br,EZEQUIAS NASCIMENTO DOS SANTOS,ADMINISTR,None,None
753,357138,01608379000180,UNIMED CENTRO OESTE PAULISTA - FEDERAÇÃO INTRA...,UNIMED CENTRO OESTE PAULISTA - FEDERACAO INTRA...,Cooperativa Médica,RUA RIO BRANCO,27-65,,JARDIM PAULISTA,Bauru,SP,17017220,14,21061400,,federacao.regional@unimedcop.coop.br,FRANCISCO VENDITTO SOARES,DIRETOR PRESIDENTE,5,None
898,324213,09237009000195,UNIMED NORTE/NORDESTE-FEDERAÇÃO INTERFEDERATIV...,UNIMED NORTE/NORDESTE,Cooperativa Médica,AV. CARNEIRO DA CUNHA,64,,TORRE,João Pessoa,PB,58040240,83,30482500,30482750,operadora@unimed-nne.com.br,VICENTE JUSTINIANO BARBOSA NETO,REPRESENTANTE LEGAL,None,None
965,359289,16991945000152,UNIMED VALE DO AÇO COOPERATIVA DE TRABALHO MÉDICO,UNIMED VALE DO AÇO COOPERATIVA DE TRABALHO MÉDICO,Cooperativa Médica,Rubens Siqueira Maia,2030,Anexo Operadora,Bairro Centro,Coronel Fabriciano,MG,35170460,31,21362287,21362201,regulatorio@unimedvaledoaco.coop.br,ERICO RAIMUNDO GUIMARAES FANTINI,DIRETOR PRESIDENTE,None,None
970,334511,01773319000112,UNIMED VALE DO PARAÍBA - FEDERAÇÃO INTRAFEDERA...,UNIMED - INTRAFEDERATIVA DO VALE DO PARAÍBA,Cooperativa Médica,RUA VISCONDE DE PINDAMONHANGABA,100,,JARDIM BOA VISTA,Pindamonhangaba,SP,12401011,12,21262400,,diretoria@unimedvaledoparaiba.coop.br,JULIO CESAR TEIXEIRA AMADO,DIRETOR PRESIDENTE,None,None
981,393321,42163881000101,UNIMED-RIO COOPERATIVA DE TRABALHO MEDICO DO R...,UNIMED RIO COOPERATIVA DE TRABALHO MEDICO DO R...,Cooperativa Médica,AV. AYRTON SENNA,2500,"BL 01, SL 404/408, BL 03, SL 101/109, SS",BARRA DA TIJUCA,Rio de Janeiro,RJ,22775003,21,31397999,31397893,controlad@unimedrio.com.br,DENISE DE ABREU DURÃO,DIRETOR,None,None
1002,400556,16325896000119,UNIODONTO DE FEIRA DE SANTANA - COOPERATIVA DE...,UNIODONTO FEIRA DE SANTANA,Cooperativa odontológica,RUA CASTRO ALVES,1511,"2º andar , sala 208 - Edifício Meritum C",CENTRO,Feira de Santana,BA,44001184,,,,uniodontofs@ig.com.br,CAMILA DOREA FERNANDES SILVA,DIRETORA PRES,None,None
1015,336017,00172586000171,UNIODONTO DE PRESIDENTE PRUDENTE COOPERATIVA O...,UNIODONTO DE PRESIDENTE PRUDENTE COOPERATIVA O...,Cooperativa odontológica,AVENIDA CORONEL JOSÉ SOARES MARCONDES,454,,VILA MACHADINHO,Presidente Prudente,SP,19020120,18,39017764,39017765,uniodonto@uniodontoprudente.com.br,EDUARDO MITSUO OTIAI,DIRETOR,None,None


In [24]:
oper_ativas['Data_Registro_ANS'].fillna('1800-01-01', inplace = True)

C:\Users\jeffe\AppData\Local\Temp\ipykernel_11504\2315252016.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  oper_ativas['Data_Registro_ANS'].fillna('1800-01-01', inplace = True)


In [25]:
oper_ativas[oper_ativas['Data_Registro_ANS'].str.contains('2017-01')]

,Registro_ANS,CNPJ,Razao_Social,Nome_Fantasia,Modalidade,Logradouro,Numero,Complemento,Bairro,Cidade,UF,CEP,DDD,Telefone,Fax,Endereco_eletronico,Representante,Cargo_Representante,Regiao_de_Comercializacao,Data_Registro_ANS
132,420794,23779545000143,ASSOREL BRASIL ADMINISTRADORA DE BENEFICIOS LT...,ASSOREL BRASIL,Administradora de Benefícios,AVENIDA HISTORIADOR RUBENS DE MENDONÇA,1856,"14º ANDAR, SALA 1404",BOSQUE DA SAUDE,Cuiabá,MT,78050000,65,30272470,,carolline@assorelbrasil.com.br,CAROLLINE BRONZEADO DE OLIVEIRA MORILHAS,SÓCIA ADMINISTRADORA,5,2017-01-
145,420697,25313773000159,BEM ESTAR ADMINISTRADORA DE BENEFÍCIOS LTDA,,Administradora de Benefícios,RUA FELIPE SCHMIDT,679,,CENTRO,Florianópolis,SC,88010000,,,,CONTATO@BEMESTARADM.COM.BR,RAQUEL VITÓRIA MATOS,ADMINISTRADORA,3,2017-01-13
335,420701,25063964000100,FUNDAÇÃO DE ASSISTÊNCIA À SAÚDE DA ASSOCIAÇÃO ...,FAS/AMP/RS,Autogestão,AV. AURELIANO DE FIGUEIREDO PINTO,501,,PRAIA DE BELAS,Porto Alegre,RS,90050191,51,30222434,,contato@fasamprs.com.br,CLAUDIO BONATTO,DIRETOR PRESIDENTE,3,2017-01-27
385,420751,26032244000140,IDEAL SAÚDE ASSISTÊNCIA MÉDICA AMBULATORIAL LTDA,IDEAL SAÚDE,Medicina de Grupo,Setor SCIA Quadra 14 Conjunto 3,LOTE 03,"Zona Industrial (Guará),",GUARA,Brasília,DF,71250115,61,32421250,,diretoria@planoidealsaude.com.br,MARYEL MATOS RODRIGUES,SÓCIO ADMINISTRADOR,4,2017-01-18
534,420778,14938785000152,PASSPORT SYSTEM ODONTOLOGIA LTDA,PASSPORT SYSTEM,Odontologia de Grupo,Avenida Rio Branco,2672,Sala A,Setor 05,Jaru,RO,76890000,69,35211342,,nhyll_odonto@hotmail.com,NILCELIA ANA MARIN,Administrador,3,2017-01-18


In [26]:
oper_ativas.loc[oper_ativas['Data_Registro_ANS'] == '2017-01-', ['Data_Registro_ANS']] = '2017-01-01'

In [27]:
oper_ativas['Data_Registro_ANS'] = pd.to_datetime(oper_ativas['Data_Registro_ANS'], errors = 'coerce')

In [28]:
# data do ultimo registro na tabela demonsta atualização recente

oper_ativas[['Data_Registro_ANS']].describe(include = 'all')

,Data_Registro_ANS
count,1103
mean,2004-03-12 04:43:18.005439744
min,1800-01-01 00:00:00
25%,1998-12-21 00:00:00
50%,1999-02-18 00:00:00
75%,2014-03-31 12:00:00
max,2025-01-27 00:00:00


In [29]:
oper_ativas.to_csv('../01_databases/operadora_ativas_ans_03022025.csv')
dicio.to_csv('../01_databases/dicionario_operadora_ativas_ans_03022025.csv')

In [30]:
ftp.close()